# Darcy equation with TPFA

In this tutorial we present how to solve a Darcy equation with [PyGeoN](https://github.com/compgeo-mox/pygeon) using the two-point flux approximation (TPFA) finite volume method. The unknown is the cell-centre pressure $p$, from which the flux $q = -K \nabla p$ can be recovered.

Let $\Omega=(0,1)^2$ with boundary $\partial \Omega$ and outward unit normal ${\nu}$. Given $K$ the permeability tensor, we want to solve the following problem: find $p$ such that
$$
-\nabla \cdot (K \nabla p) = 0 \quad \text{in } \Omega
$$
with boundary conditions:
$$ p = 1 \text{ on } \partial_{\rm bottom} \Omega \qquad p = 0 \text{ on } \partial_{\rm top} \Omega \qquad \nu \cdot q = 0 \text{ on } \partial_{\rm left} \Omega \cup \partial_{\rm right} \Omega$$

For $K = I$ the exact solution is the linear profile $p = 1 - y$.

We present *step-by-step* how to create the grid, declare the problem data, and finally solve the problem.

First we import some of the standard modules. Since PyGeoN is based on [PorePy](https://github.com/pmgbergen/porepy) we import both modules.

In [22]:
import os

import numpy as np
import porepy as pp

import pygeon as pg

We create now the grid. TPFA works on general polygonal/polyhedral meshes; here we use a Cartesian grid for simplicity. In this example we consider a 2-dimensional structured grid, but the presented code will work also in 1d and 3d.

In [23]:
n = 10
dim = 2

sd = pp.CartGrid([n] * dim, [1] * dim)
sd = pg.convert_from_pp(sd)
sd.compute_geometry()

With the following code we set the data, in particular the permeability tensor, and the boundary conditions. Since we need to identify each side of $\partial \Omega$ we need a few steps.

TPFA uses the `pg.FlowBC` object to handle boundary conditions in a unified way. Pressure boundary conditions are set with `set_pressure_bcs` and flux conditions with `set_flux_bcs`. Any face not explicitly assigned a condition defaults to zero flux.

In [ ]:
key = "darcy_tpfa"

perm = pp.SecondOrderTensor(np.ones(sd.num_cells))
param = {pg.SECOND_ORDER_TENSOR: perm}
data = pp.initialize_data({}, key, param)

# identify the boundary portions
bottom = np.isclose(sd.face_centers[1, :], 0)
top = np.isclose(sd.face_centers[1, :], 1)
left = np.isclose(sd.face_centers[0, :], 0)
right = np.isclose(sd.face_centers[0, :], 1)

# create the BC object (also registers itself in data)
bcs = pg.FlowBC(sd, data, key)

# pressure conditions: p = 1 at the bottom, p = 0 at the top
bcs.set_pressure_bcs(bottom, np.ones(sd.num_faces))
bcs.set_pressure_bcs(top)

# no-flow conditions on the left and right sides
bcs.set_flux_bcs(left)
bcs.set_flux_bcs(right)

Once the data are assigned to the grid, we assemble the system matrix and the boundary right-hand side with `pg.TPFA`. The degrees of freedom are the cell-centre pressure values, so the system has $N_c$ unknowns.

In [25]:
tpfa = pg.TPFA(key)

A = tpfa.assemble_system_matrix(sd, data)
rhs = tpfa.assemble_rhs_boundary_vector(sd, data)

We solve the linear system with `pg.LinearSystem` to obtain the cell-centre pressures.

In [26]:
ls = pg.LinearSystem(A, rhs)
p = ls.solve()

We finally export the solution to be visualized by [ParaView](https://www.paraview.org/).

In [27]:
folder_name = os.path.join(os.getcwd(), key)
file_name = "sol"

save = pp.Exporter(sd, file_name, folder_name=folder_name)
save.write_vtu([("cell_p", p)])

A representation of the computed solution is given below, where the cells are colored with $p$.

<div style="text-align: center;">
  <img src="./fig/darcy_tpfa.png" alt="TPFA Darcy solution"/>
</div>

In [28]:
# Consistency check: TPFA recovers the exact linear solution p = 1 - y
p_exact = tpfa.interpolate(sd, lambda x: 1 - x[1])

assert np.isclose(np.linalg.norm(p), 5.766281297335398)
assert np.allclose(p, p_exact)